# Nondeterministic Actions: Solving Tic-Tac-Toe with AND-OR-Tree Search

## Introduction 
 
Multiplayer games can be implemented as:
1. __Nondeterministic actions:__ The opponent is seen as part of an environment with nondeterministic actions. Non-determinism is the result of the unknown opponent's moves. 
2. Optimal Decisions: Minimax search (search complete game tree) and alpha-beta pruning.
3. Heuristic Alpha-Beta Tree Search: Cut off tree search and use heuristic to estimate state value. 
4. Monte Carlo Tree search: Simulate playouts to estimate state value. 

Here we will implement search for Tic-Tac-Toe (see [rules](https://en.wikipedia.org/wiki/Tic-tac-toe)).

We will implement
* __AND-OR-Tree search.__

Each action consists of the move by the player (OR levels in the tree) and all possible (i.e., nondeterministic) responses by the opponents (AND levels). The action therefore results in a set of possible states.

We will search for a __conditional plan__ using AND-OR-Tree search. 

The code for the basic functions used for search are implemented in [tictactoe.py](tictactoe.py).

In [1]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("tictactoe.py", ".")

In [2]:
from tictactoe import empty_board, actions, result, terminal, utility, other, show_board
from tictactoe import random_player, play 

## Solving Tic-Tac-Toe using AND-OR-Tree Search

The opponent can be modeled as part of the environment. The opponent's action as a response to the agent's 
action makes the environment appear non-deterministic (see AIMA page 125 or the slides). The ADS-OR-Tree algorithm uses the following functions:

* `is_goal(state)`: determine if the current board/state has a winner.
* `is_cycle(path)`: determine if we are in a loop (the tic-tac-toe problem has no loops so I do not implement it).
* `results(state, action)`: returns all states following state + action.


### Cycles

The path is typically used for loop checking. We ignore it here since tic-tac-toe does not have cycles.

### Goals

Reaching the goal means that the agent has successfully finished the task. 

For games, `terminal(s)` checks if `s` is a terminal state, but a terminal state can also indicate that 
the player lost (which indicates a failure). This can determined by the 
`utility(s)` function. Tic-tac-toe also has draws and we need to decide if a draw is considered successfully reaching
the goal or a failure.

### Non-deterministic Results

Since the opponent and her moves are part of the environment, `results()` returns for a state $\times$ action combination a set of new states that reflects all possible responses by the opponent.

In [3]:
def results(state, action, player = 'x'):
    """produce the set of states after the provided action for player. 
       It is the set of boards with the action and all possible reactions by the opponent."""
    
    opponent = other(player)
    state = state.copy()
    
    # player's move
    state[action] = player
    
    # opponent reacts
    r = list()
    o_actions = actions(state)
    
    # board is full
    if len(o_actions) < 1 : return [state]
    
    for o_a in o_actions:
        s = state.copy()
        s[o_a] = opponent
        r.append(s)    
    
    return r

show_board(empty_board())

print()
print("Set of possible state for placing an x at position 4 of an empty board and the opponent's move:")
results(empty_board(), 4)


Set of possible state for placing an x at position 4 of an empty board and the opponent's move:


[['o', ' ', ' ', ' ', 'x', ' ', ' ', ' ', ' '],
 [' ', 'o', ' ', ' ', 'x', ' ', ' ', ' ', ' '],
 [' ', ' ', 'o', ' ', 'x', ' ', ' ', ' ', ' '],
 [' ', ' ', ' ', 'o', 'x', ' ', ' ', ' ', ' '],
 [' ', ' ', ' ', ' ', 'x', 'o', ' ', ' ', ' '],
 [' ', ' ', ' ', ' ', 'x', ' ', 'o', ' ', ' '],
 [' ', ' ', ' ', ' ', 'x', ' ', ' ', 'o', ' '],
 [' ', ' ', ' ', ' ', 'x', ' ', ' ', ' ', 'o']]

### Recursive AND-OR Search (DFS)

In [4]:
# define global variables for debugging
DEBUG = 1
COUNT = 0 # used to report the number of searched nodes

def and_or_search(state, player = 'x', draw_is_win = True):
    """start the search. Consider draw_is_win a goal state?"""
    global DEBUG, COUNT
    COUNT = 0
    
    plan = or_search(state, player, draw_is_win)
    
    if DEBUG >= 1: 
        print(f"Number of nodes searched: {COUNT}")  
    
    return plan
 

def or_search(state, player, draw_is_win):
    """Or step of the search: the player makes a move. 
    We try all possible action and return a conditional 
    plan for the first action that only has goal states as leaf nodes. 
    If none can be found, then failure (None) is returned.
    
    Note: path is not used in this implementation since tic-tac-toe does not 
    need cycle checking.
    """
    global DEBUG, COUNT
    COUNT += 1
    
    # goal check
    u = utility(state, player)
    if u == -1 or (not draw_is_win and u == 0): return None # failure
     
    # cycles check: tic-tac-toe has no cycles

    # check all possible actions
    for action in actions(state):
        plan = and_search(results(state, action, player), player, draw_is_win)
        if plan is not None: 
            return [action, plan]
    
    # failure
    return None


def and_search(states, player, draw_is_win):
    """And step of the search: Represents all opponent's possible moves. 
    Follow all possible states (call the or step). 
    Return a conditional plan only if all paths lead to a goal state."""
    global DEBUG, COUNT
    COUNT += 1
    
    # return plans only if no state fails
    plans = []
    for s in states:
        # goal check
        u = utility(s, player)
        if u == 1: return "win"  # success
        if draw_is_win and u == 0: return "draw"

        plan = or_search(s, player, draw_is_win)
        
        if plan is None: 
            return None    # found a state that fails so we abandon this subtree!
        plans.append(['if', s, 'then', plan])
        
    return plans

And-or search looks for a subtree (i.e., an action) that guarantees a win and returns a conditional plan for this subtree. If no such subtree exists, no plan is returned. The algorithm only searches a fraction of the game tree since it abandons the current subtree when it finds the first leaf node that is not a win (see `and_search` function).

# Some Tests

## x is about to win

In [5]:
board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'

print("Board:")
show_board(board)

print()
print("Win or draw:")
%time display(and_or_search(board, player = 'x', draw_is_win = True))

Board:

Win or draw:
Number of nodes searched: 22


[2,
 [['if', ['x', 'o', 'x', 'o', 'x', 'o', ' ', ' ', ' '], 'then', [6, 'win']],
  ['if',
   ['x', 'o', 'x', 'o', 'x', ' ', 'o', ' ', ' '],
   'then',
   [5,
    [['if', ['x', 'o', 'x', 'o', 'x', 'x', 'o', 'o', ' '], 'then', [8, 'win']],
     ['if',
      ['x', 'o', 'x', 'o', 'x', 'x', 'o', ' ', 'o'],
      'then',
      [7, 'draw']]]]],
  ['if',
   ['x', 'o', 'x', 'o', 'x', ' ', ' ', 'o', ' '],
   'then',
   [5,
    [['if', ['x', 'o', 'x', 'o', 'x', 'x', 'o', 'o', ' '], 'then', [8, 'win']],
     ['if',
      ['x', 'o', 'x', 'o', 'x', 'x', ' ', 'o', 'o'],
      'then',
      [6, 'win']]]]],
  ['if',
   ['x', 'o', 'x', 'o', 'x', ' ', ' ', ' ', 'o'],
   'then',
   [5,
    [['if',
      ['x', 'o', 'x', 'o', 'x', 'x', 'o', ' ', 'o'],
      'then',
      [7, 'draw']],
     ['if',
      ['x', 'o', 'x', 'o', 'x', 'x', ' ', 'o', 'o'],
      'then',
      [6, 'win']]]]]]]

CPU times: user 3.48 ms, sys: 41 μs, total: 3.52 ms
Wall time: 3.37 ms


Why does it play 2 and not 8!!!

In [6]:
print("Win only:")
%time display(and_or_search(board, player = 'x', draw_is_win = False))

Win only:
Number of nodes searched: 30


[2,
 [['if', ['x', 'o', 'x', 'o', 'x', 'o', ' ', ' ', ' '], 'then', [6, 'win']],
  ['if', ['x', 'o', 'x', 'o', 'x', ' ', 'o', ' ', ' '], 'then', [8, 'win']],
  ['if',
   ['x', 'o', 'x', 'o', 'x', ' ', ' ', 'o', ' '],
   'then',
   [5,
    [['if', ['x', 'o', 'x', 'o', 'x', 'x', 'o', 'o', ' '], 'then', [8, 'win']],
     ['if',
      ['x', 'o', 'x', 'o', 'x', 'x', ' ', 'o', 'o'],
      'then',
      [6, 'win']]]]],
  ['if', ['x', 'o', 'x', 'o', 'x', ' ', ' ', ' ', 'o'], 'then', [6, 'win']]]]

CPU times: user 2.33 ms, sys: 1.03 ms, total: 3.36 ms
Wall time: 3.09 ms


## x can draw_is_win if it chooses 7

In [7]:

board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[2] = 'x'
#board[3] = 'o'
board[4] = 'o'

print("Board:")
show_board(board)

print()
print("Win or draw:")
%time display(and_or_search(board, player = 'x', draw_is_win = True))

Board:

Win or draw:
Number of nodes searched: 64


[7,
 [['if',
   ['x', 'o', 'x', 'o', 'o', ' ', ' ', 'x', ' '],
   'then',
   [5,
    [['if', ['x', 'o', 'x', 'o', 'o', 'x', 'o', 'x', ' '], 'then', [8, 'win']],
     ['if',
      ['x', 'o', 'x', 'o', 'o', 'x', ' ', 'x', 'o'],
      'then',
      [6, 'draw']]]]],
  ['if',
   ['x', 'o', 'x', ' ', 'o', 'o', ' ', 'x', ' '],
   'then',
   [3,
    [['if',
      ['x', 'o', 'x', 'x', 'o', 'o', 'o', 'x', ' '],
      'then',
      [8, 'draw']],
     ['if',
      ['x', 'o', 'x', 'x', 'o', 'o', ' ', 'x', 'o'],
      'then',
      [6, 'win']]]]],
  ['if',
   ['x', 'o', 'x', ' ', 'o', ' ', 'o', 'x', ' '],
   'then',
   [3,
    [['if',
      ['x', 'o', 'x', 'x', 'o', 'o', 'o', 'x', ' '],
      'then',
      [8, 'draw']],
     ['if',
      ['x', 'o', 'x', 'x', 'o', ' ', 'o', 'x', 'o'],
      'then',
      [5, 'draw']]]]],
  ['if',
   ['x', 'o', 'x', ' ', 'o', ' ', ' ', 'x', 'o'],
   'then',
   [3,
    [['if', ['x', 'o', 'x', 'x', 'o', 'o', ' ', 'x', 'o'], 'then', [6, 'win']],
     ['if',
      ['x', '

CPU times: user 4.02 ms, sys: 1.01 ms, total: 5.03 ms
Wall time: 4.8 ms


In [8]:
print("Win only:")
%time display(and_or_search(board, player = 'x', draw_is_win = False))

Win only:
Number of nodes searched: 69


None

CPU times: user 3.01 ms, sys: 0 ns, total: 3.01 ms
Wall time: 2.83 ms


## a bad situation

In [9]:
board = empty_board() 
board[0] = 'o'
board[2] = 'x'
board[8] = 'o'

print("Board:")
show_board(board)

print()
print("Win or draw:")
%time display(and_or_search(board, player = 'x', draw_is_win = True))

Board:

Win or draw:
Number of nodes searched: 155


None

CPU times: user 5.31 ms, sys: 944 μs, total: 6.25 ms
Wall time: 6.01 ms


## Empty board: Only a draw_is_win can be guaranteed

In [10]:
board = empty_board() 

print("Board:")
show_board(board)

print()
print("Win or draw:")
%time display(and_or_search(board, player = 'x', draw_is_win = True))

Board:

Win or draw:
Number of nodes searched: 966


[0,
 [['if',
   ['x', 'o', ' ', ' ', ' ', ' ', ' ', ' ', ' '],
   'then',
   [2,
    [['if',
      ['x', 'o', 'x', 'o', ' ', ' ', ' ', ' ', ' '],
      'then',
      [4,
       [['if',
         ['x', 'o', 'x', 'o', 'x', 'o', ' ', ' ', ' '],
         'then',
         [6, 'win']],
        ['if',
         ['x', 'o', 'x', 'o', 'x', ' ', 'o', ' ', ' '],
         'then',
         [5,
          [['if',
            ['x', 'o', 'x', 'o', 'x', 'x', 'o', 'o', ' '],
            'then',
            [8, 'win']],
           ['if',
            ['x', 'o', 'x', 'o', 'x', 'x', 'o', ' ', 'o'],
            'then',
            [7, 'draw']]]]],
        ['if',
         ['x', 'o', 'x', 'o', 'x', ' ', ' ', 'o', ' '],
         'then',
         [5,
          [['if',
            ['x', 'o', 'x', 'o', 'x', 'x', 'o', 'o', ' '],
            'then',
            [8, 'win']],
           ['if',
            ['x', 'o', 'x', 'o', 'x', 'x', ' ', 'o', 'o'],
            'then',
            [6, 'win']]]]],
        ['if',
        

CPU times: user 49.1 ms, sys: 975 μs, total: 50.1 ms
Wall time: 49.9 ms


In [11]:
print("Win only:")
%time display(and_or_search(board, player = 'x', draw_is_win = False))

Win only:


Number of nodes searched: 16213


None

CPU times: user 461 ms, sys: 2.72 ms, total: 464 ms
Wall time: 463 ms


## Experiments

### And-Or Tree Search vs. Random

Put AND-OR search into a wrapper agent function. Note that if AND-OR search cannot guarantee a win then
it does not return a plan, nut `None`. In this case, we do not know what the best (i.e., the least "bad") 
move is, so I make the agent play randomly. Other methods that we will learn about later can determine the utility of a move, even if it does not guarantee a win.

In [12]:
import numpy as np

DEBUG = 0

def and_or_player(board, player='x'):
    plan = and_or_search(board, player)
    if plan is None:
        return np.random.choice(actions(board))
    return plan[0]


N_GAMES = 100
SEED = 20261110

np.random.seed(SEED)
print(f'and-or-search (X) vs random (O), {N_GAMES} games:')
display(play(and_or_player, random_player, N=N_GAMES))

np.random.seed(SEED)
print(f'random (X) vs and-or-search (O), {N_GAMES} games:')
display(play(random_player, and_or_player, N=N_GAMES))

and-or-search (X) vs random (O), 100 games:


{'x': 95, 'o': 0, 'd': 5}

random (X) vs and-or-search (O), 100 games:


{'x': 0, 'o': 72, 'd': 28}

**Note how slow this is!**

My lazy implementation is a _simple reflex agent_ that recreates a conditional plan every time it is asked for an action. ut since And-Or search already produces a complete conditional plan for the empty board, a better implementation would use a _model-based reflex agent_ that store this precomputed plan and just follow it without more planing.

### And-Or Tree Search vs. And-Or Tree Search

In [13]:
# No randomness -> run only once

%time display(play(and_or_player, and_or_player, N = 1))

{'x': 0, 'o': 0, 'd': 1}

CPU times: user 68.2 ms, sys: 0 ns, total: 68.2 ms
Wall time: 67.8 ms


## Results and explanation

Board cells are numbered 0-8 from left to right, top to bottom. `results(state, action, player)` applies the player's move, then returns one board for every legal opponent reply (the nondeterministic outcomes); if there is no reply, it returns the resulting board. `and_or_search` starts the search and resets the node count. `or_search` is the player's choice: it tries legal moves and succeeds if it finds an action whose `and_search` succeeds. `and_search` checks the opponent's possible replies and builds `['if', board, 'then', plan]` branches; every branch must lead to a goal. With `draw_is_win=True`, both a win and a draw count as successful leaves.

### Existing test cases

- **X is about to win** (`X` at 0,4; `O` at 1,3): the first action is **2**. The conditional plan responds to `O:5` with `X:6`; for `O:6`, `O:7`, or `O:8`, it responds with `X:5`, then continues according to the resulting board. The plan's terminal leaves are 5 wins and 2 draws, so it avoids a loss but is not a forced win in every branch. With `draw_is_win=False`, the first action is still 2.
- **X needs at least a draw** (`X` at 0,2; `O` at 1,4): the first action is **7**. If O replies at 3, X plays 5; if O replies at 5, 6, or 8, X plays 3, then follows the conditional branches. The plan has 3 winning and 5 drawing leaves. With `draw_is_win=False`, no plan is found, consistent with this being a draw-seeking case.
- **Unfavorable position** (`O` at 0,8; `X` at 2): no plan is found even when draws count as success, so the wrapper player falls back to a random legal move.
- **Empty board**: the first action is **0** (a corner). Its first conditional replies are `O:1 -> X:2`, `O:2 -> X:3`, `O:3 -> X:1`, `O:4 -> X:1`, `O:5 -> X:2`, `O:6 -> X:1`, `O:7 -> X:1`, and `O:8 -> X:2`; later moves are also conditional. There are 98 winning and 26 drawing plan leaves. With `draw_is_win=False`, no plan is found.

The leaf counts describe branches in the returned search plan, not outcome probabilities.

### 100-game match results

Using `N_GAMES = 100` and NumPy seed `20261110` (counts are wins for X, wins for O, draws):

- AND-OR as X vs. random as O: **X 95, O 0, draws 5**.
- Random as X vs. AND-OR as O: **X 0, O 72, draws 28**.

The AND-OR agent searches a fresh conditional plan for each turn; when no win-or-draw plan exists, it selects a random legal move.

### Discussion 1 — Minimax và AND-OR Search (Hồ Minh Tiến)

Minimax và AND-OR đều xét các phản ứng của đối thủ, nhưng khác cách biểu diễn kết quả. Minimax luân phiên MAX/MIN để trả về **giá trị tối ưu** (+1 thắng, 0 hòa, -1 thua), còn AND-OR tạo **kế hoạch có điều kiện**: nút OR chọn nước đi, nút AND phải xử lý được mọi phản ứng hợp lệ của đối thủ.

Trong code, `results(state, action, player)` sinh các trạng thái sau nước đi của ta và mọi phản ứng của đối thủ. `or_search` chọn hành động có `and_search` thành công; `and_search` chỉ chấp nhận khi các nhánh đều đạt mục tiêu. Với `draw_is_win=True`, hòa được xem là thành công; nếu chỉ chấp nhận thắng, `and_or_search` trả `None` trên bàn trống. Minimax vẫn xác định được giá trị tối ưu của bàn trống là hòa (0), kể cả khi không thể ép thắng.

Kết quả đấu Random được chạy ngay trong notebook này (100 ván mỗi lượt, seed 20261110). Các số thắng/hòa/thua mô tả **mẫu đã chạy**, không phải đảm bảo xác suất cho mọi ván. Ô kiểm chứng bên dưới đối chiếu giá trị Minimax độc lập với khả năng tìm kế hoạch của AND-OR và chạy lại hai loạt trận để báo số liệu tái lập.

In [14]:
# Đối chứng cho Discussion 1: Minimax giá trị tối ưu vs kế hoạch AND-OR.
from functools import lru_cache

@lru_cache(maxsize=None)
def reference_minimax_value(cells, to_move):
    state = list(cells)
    outcome = utility(state, 'x')
    if outcome is not None:
        return outcome
    child_values = [
        reference_minimax_value(tuple(result(state, to_move, a)), other(to_move))
        for a in actions(state)
    ]
    return max(child_values) if to_move == 'x' else min(child_values)

initial = empty_board()
minimax_empty_value = reference_minimax_value(tuple(initial), 'x')
DEBUG = 0
conditional_non_loss_plan = and_or_search(initial, player='x', draw_is_win=True)
conditional_forced_win_plan = and_or_search(initial, player='x', draw_is_win=False)
assert minimax_empty_value == 0
assert conditional_non_loss_plan is not None
assert conditional_forced_win_plan is None
print('Empty board: Minimax utility =', minimax_empty_value)
print('AND-OR win-or-draw plan exists:', conditional_non_loss_plan is not None)
print('AND-OR forced-win plan exists:', conditional_forced_win_plan is not None)
print('Unique Minimax cache entries:', reference_minimax_value.cache_info().currsize)

np.random.seed(20261110)
and_or_first = play(and_or_player, random_player, N=100)
np.random.seed(20261110)
and_or_second = play(random_player, and_or_player, N=100)
assert sum(and_or_first.values()) == sum(and_or_second.values()) == 100
print('AND-OR first vs Random (X wins / O wins / draws):', and_or_first)
print('AND-OR second vs Random (X wins / O wins / draws):', and_or_second)

Empty board: Minimax utility = 0
AND-OR win-or-draw plan exists: True
AND-OR forced-win plan exists: False
Unique Minimax cache entries: 5478


AND-OR first vs Random (X wins / O wins / draws): {'x': 95, 'o': 0, 'd': 5}
AND-OR second vs Random (X wins / O wins / draws): {'x': 0, 'o': 72, 'd': 28}
